# 01 First calls
Example notebook for stage 1. Run each cell in order.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-01"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: load your key

In [ ]:
from kit.llm import call_llm, pick_provider, LLMError
from kit.prices import cost
import pandas as pd

PROVIDER = pick_provider()  # gemini if GEMINI_API_KEY is set, else openai_compatible, mock if KIT_OFFLINE=1
print("Provider:", PROVIDER)
if PROVIDER == "mock":
    print("Warning: this is the offline mock model. Its outputs are not real runs, so do not submit them.")

Step 3: set the prompt and settings by hand

In [ ]:
SYSTEM = "You are the help assistant for Quillnote, a note app. Answer in at most 3 short steps."
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 200
MODEL = None  # None means the provider default (see kit/llm.py)

QUESTIONS = [
    "How do I export a note as PDF?", "My notes stopped syncing", "Can I share a notebook with my team?",
    "How do I undo a deleted note?", "Where is dark mode?", "Can I search inside images?",
    "Why was I charged twice?", "How do I add a tag?", "The app crashes when I open a big note",
    "Can I use Quillnote offline?",
]

Step 4: send 10 requests

In [ ]:
results = []
for q in QUESTIONS:
    r = call_llm(q, system=SYSTEM, model=MODEL, temperature=TEMPERATURE, max_output_tokens=MAX_OUTPUT_TOKENS)
    results.append(r)
    print(f"{q[:40]:40} -> {r['text'][:70]!r}")

Step 5: read the usage block

In [ ]:
usage = pd.DataFrame([{
    "question": q, "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
    "stop_reason": r["stop_reason"], "latency_s": r["latency_s"],
    "cost_usd": cost(r["input_tokens"], r["output_tokens"], r["model"]),
} for q, r in zip(QUESTIONS, results)])
print("Model:", results[0]["model"])
usage

In [ ]:
print("Total cost: $%.5f" % usage.cost_usd.sum())
print("Median latency: %.2fs" % usage.latency_s.median())
usage[["input_tokens", "output_tokens"]].describe()

Step 6: force a truncation with a low output cap

In [ ]:
short = call_llm(QUESTIONS[0], system=SYSTEM, model=MODEL, temperature=TEMPERATURE, max_output_tokens=10)
print("Stop reason:", short["stop_reason"], "| output tokens:", short["output_tokens"])
print("Text:", repr(short["text"]))

Step 7: trigger a 400 error on purpose

In [ ]:
try:
    call_llm(QUESTIONS[0], system=SYSTEM, model=MODEL, temperature=5)  # temperature must be 0 to 2
except LLMError as e:
    print("Caught error. Status:", e.status)
    print("Message:", e.message[:300])

Step 8: see what a 429 looks like (simulated)

In [ ]:
# A real 429 means you sent too many requests. call_llm retries with longer waits, then gives up.
# This uses the mock provider so it never touches your real quota.
try:
    call_llm("SIMULATE_429", provider="mock")
except LLMError as e:
    print("Caught error. Status:", e.status, "|", e.message)
    print("429 = rate limit. Wait and retry, slow down, or check your quota.")

Step 9: next, open the 01 probe set notebook

Your five predicted weaknesses and your short README section go at the end of that notebook, so the whole stage can be read in two notebooks.